# Speech to Text — Audio File Input

This notebook recognizes speech from a local **WAV audio file** and demonstrates how to retrieve detailed recognition results including word-level timing information.

By default the notebook downloads a sample WAV file. You can replace `AUDIO_FILE` with the path to any mono WAV file sampled at 16 kHz.

In [ ]:
%pip install azure-cognitiveservices-speech python-dotenv requests --quiet

In [ ]:
import os
import json
import requests
import azure.cognitiveservices.speech as speechsdk
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from repo root

speech_key = os.environ["FOUNDRY_AI_SERVICES_KEY"]
speech_region = os.environ["FOUNDRY_AI_SERVICES_REGION"]

print("Environment loaded.")

In [ ]:
# Download a sample WAV file if it doesn't already exist
AUDIO_FILE = "sample_audio.wav"
SAMPLE_URL = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-speech-sdk/master/samples/python/console/whatstheweatherlike.wav"

if not os.path.exists(AUDIO_FILE):
    print(f"Downloading sample audio to {AUDIO_FILE} ...")
    audio_data = requests.get(SAMPLE_URL).content
    with open(AUDIO_FILE, "wb") as f:
        f.write(audio_data)
    print("Download complete.")
else:
    print(f"Using existing file: {AUDIO_FILE}")

In [ ]:
# Configure speech recognition with detailed output and word-level timestamps
speech_config = speechsdk.SpeechConfig(subscription=speech_key, region=speech_region)
speech_config.request_word_level_timestamps()
speech_config.speech_recognition_language = "en-US"

audio_config = speechsdk.audio.AudioConfig(filename=AUDIO_FILE)
recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_config)

print(f"Recognizing speech from: {AUDIO_FILE}")
result = recognizer.recognize_once_async().get()

if result.reason == speechsdk.ResultReason.RecognizedSpeech:
    print(f"\nRecognized: {result.text}")
elif result.reason == speechsdk.ResultReason.NoMatch:
    print(f"No speech could be recognized: {result.no_match_details}")
elif result.reason == speechsdk.ResultReason.Canceled:
    details = result.cancellation_details
    print(f"Recognition canceled: {details.reason}")
    if details.reason == speechsdk.CancellationReason.Error:
        print(f"Error details: {details.error_details}")

In [ ]:
# Parse and display detailed results including word-level timing
# Time units are 100-nanosecond ticks; divide by 10_000 to get milliseconds
if result.reason == speechsdk.ResultReason.RecognizedSpeech:
    json_result = json.loads(result.json)
    best = json_result["NBest"][0]

    print("\n=== Detailed Results ===")
    print(f"Lexical   : {best['Lexical']}")
    print(f"ITN       : {best['ITN']}")
    print(f"Display   : {best['Display']}")
    print(f"Confidence: {best['Confidence']:.4f}")

    print("\nWord-level timing (ms):")
    print(f"{'Word':<20} {'Offset (ms)':>12} {'Duration (ms)':>14}")
    print("-" * 48)
    for word in best.get("Words", []):
        offset_ms = word["Offset"] / 10_000
        duration_ms = word["Duration"] / 10_000
        print(f"{word['Word']:<20} {offset_ms:>12.1f} {duration_ms:>14.1f}")